# (0) MLP: A * B % C (A,B between 0..99, C between 0..19)

In [ ]:
#@title (1) setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import itertools, random, time
import copy

# Reproducibility
SEED = 42
def resetRandom():
  torch.manual_seed(SEED)
  np.random.seed(SEED)
  random.seed(SEED)
resetRandom()

# training time optimization
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
#@title (2) dataset
def one_hot(n, num_classes=100):
    # One-hot encode a single integer.
    v = np.zeros(num_classes, dtype=np.float32)
    v[n] = 1.0
    return v

# ── Generate all 190,000 input pairs ──
pairs = list(itertools.product(range(100), range(100), range(19)))
random.shuffle(pairs)

X_all, y_all = [], []
for a, b, c in pairs:
    x = np.concatenate([one_hot(a), one_hot(b), one_hot(c, 19)])   # 220-dim
    y = (a * b) % (c+1)
    X_all.append(x)
    y_all.append(y)

X_all = np.array(X_all)
y_all = np.array(y_all)

# 80-20 split
split = int(0.8 * len(X_all))
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

# DataLoaders — mini-batch (full-batch would be 8K samples)
BATCH_SIZE = 256
train_ds = TensorDataset(torch.tensor(X_train).to(DEVICE), torch.tensor(y_train, dtype=torch.long).to(DEVICE))
test_ds  = TensorDataset(torch.tensor(X_test).to(DEVICE),  torch.tensor(y_test, dtype=torch.long).to(DEVICE))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

INPUT_DIM = X_train.shape[1]
OUTPUT_DIM = len(np.unique(y_all))
NUM_CLASSES = int(y_all.max()) + 1

print(f"Train: {len(train_ds):,} samples  |  Test: {len(test_ds):,} samples")
print(f"Input dim: {INPUT_DIM}  |  Output classes: {OUTPUT_DIM}")
print(f"Label distribution (train): {dict(zip(*np.unique(y_train, return_counts=True)))}")

Train: 152,000 samples  |  Test: 38,000 samples
Input dim: 219  |  Output classes: 19
Label distribution (train): {np.int64(0): np.int64(51411), np.int64(1): np.int64(11977), np.int64(2): np.int64(14363), np.int64(3): np.int64(10413), np.int64(4): np.int64(11615), np.int64(5): np.int64(6717), np.int64(6): np.int64(10053), np.int64(7): np.int64(4823), np.int64(8): np.int64(6869), np.int64(9): np.int64(4650), np.int64(10): np.int64(4769), np.int64(11): np.int64(2542), np.int64(12): np.int64(4452), np.int64(13): np.int64(1747), np.int64(14): np.int64(2023), np.int64(15): np.int64(1374), np.int64(16): np.int64(1254), np.int64(17): np.int64(538), np.int64(18): np.int64(410)}


In [ ]:
#@title (3.1) DNN: 16x2 growing & shrinking FFN w/ NormType + IsPostNmor + Residuals + Dropout
class DNN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, output_dim=OUTPUT_DIM, hidden_dim=128,
                 layer_depth=16, norm_type=nn.RMSNorm, is_post_norm=False, dropout_rate=0.0):
        super().__init__()
        self.is_post_norm = is_post_norm
        self.proj_in = nn.Linear(input_dim, hidden_dim)
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        for _ in range(layer_depth):
            self.layers.append(
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 2),
                    nn.ReLU(),
                    nn.Linear(hidden_dim * 2, hidden_dim))
                )
            if norm_type is not None:
                self.norms.append(norm_type(hidden_dim))
            else:
                self.norms.append(nn.Identity())
            self.dropouts.append(nn.Dropout(dropout_rate))
        self.proj_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.proj_in(x))
        for layer, norm, dropout in zip(self.layers, self.norms, self.dropouts):
            if self.is_post_norm:
                x = norm(x + dropout(layer(x)))
            else:
                x = x + dropout(layer(norm(x)))
        return self.proj_out(x)

MODEL = DNN().to(DEVICE)
print(MODEL)

print(f"\nTotal parameters: {sum(p.numel() for p in MODEL.parameters()):,}")
print(f"Per-layer shapes:")
for name, param in MODEL.named_parameters():
    if "weight" in name:
        print(f"  {name:30s}  {str(list(param.shape)):>15s}  ({param.numel():>6,} params)")

In [ ]:
#@title (3.2) train

all_runs = []  # global: stores {"name", "converged_epoch", "history"} per run

def evaluate(loader, model_c, criterion):
    model_c.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model_c(xb)
            total_loss += criterion(logits, yb).item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(yb)
    return total_loss / total, correct / total

def init_weights_common(m):
    if hasattr(m, 'weight') and m.weight.dim() >= 2:
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')

def train_common(model_c, run_name, lr_c=3e-3, weight_decay_c=0.1, max_epochs_c=200,
                 init_c=init_weights_common, clip_grad_c=True):
    # Reset model_c and optimizer
    resetRandom()

    model_c = torch.compile(model_c)
    if init_c != None:
        model_c.apply(init_c)

    optimizer = torch.optim.AdamW(model_c.parameters(), lr=lr_c, weight_decay=weight_decay_c)
    criterion = nn.CrossEntropyLoss()

    # Reset logging
    history = {
        "epoch": [], "train_loss": [], "test_loss": [],
        "train_acc": [], "test_acc": []
    }

    print(f"{'='*60}")
    print(f"  🚀 Run: {run_name}")
    print(f"{'='*60}")

    # ── Training loop ──
    reached_100 = False

    for epoch in range(1, max_epochs_c + 1):
        model_c.train()
        for xb, yb in train_loader:
            logits = model_c(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            if clip_grad_c:
                torch.nn.utils.clip_grad_norm_(model_c.parameters(), max_norm=1.0)
            optimizer.step()

        train_loss, train_acc = evaluate(train_loader, model_c, criterion)
        test_loss, test_acc   = evaluate(test_loader, model_c, criterion)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        print(f"Epoch {epoch:>6d} | "
              f"Train loss {train_loss:.4f}  acc {train_acc:.1%} | "
              f"Test loss {test_loss:.4f}  acc {test_acc:.1%}")

        if test_acc > 0.999 and train_acc > 0.999:
            print(f"{'='*60}")
            print(f"  ✅ 100% test accuracy reached at epoch {epoch}!")
            print(f"  📊 Final — Train: {train_acc:.2%}  Test: {test_acc:.2%}")
            print(f"{'='*60}")
            reached_100 = True
            break

    if not reached_100:
        print(f"⚠️  Did not reach 100% in {max_epochs_c} epochs.")
        print(f"    Final test acc: {history['test_acc'][-1]:.1%}")

    converged_epoch = epoch if reached_100 else None
    all_runs.append({"name": run_name,
                     "converged_epoch": converged_epoch,
                     "history": dict(history),
                     "model": copy.deepcopy(model_c)})

    return model_c


def get_history(run_name):
    for run in all_runs:
        if run["name"] == run_name:
            return run["history"]
    return None

def get_model(run_name):
    for run in all_runs:
        if run["name"] == run_name:
            return run["model"]
    return None

In [ ]:
#@title (4) visualize individual run

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors

import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# Prefer the deployment font when available, otherwise fall back to common sans-serif fonts
font_candidates = ['Inter', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
plt.rcParams['font.family'] = font_candidates
font_path = 'fonts/Inter-Regular.ttf'
try:
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
except Exception:
    pass

# Set styling for all text and lines
plt.rcParams['text.color'] = '#F1F5F9'
plt.rcParams['axes.labelcolor'] = '#F1F5F9'
plt.rcParams['xtick.color'] = '#F1F5F9'
plt.rcParams['ytick.color'] = '#F1F5F9'
plt.rcParams['axes.edgecolor'] = '#F1F5F9'

# Grid styling
plt.rcParams['grid.color'] = '#5D6674'
plt.rcParams['grid.alpha'] = 0.7

def _coplot_luminance_from_rgba(rgba):
    r, g, b = mcolors.to_rgb(rgba)
    def _linear(c):
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * _linear(r) + 0.7152 * _linear(g) + 0.0722 * _linear(b)

def _coplot_text_color_for_rgba(rgba, preferred='#F1F5F9'):
    try:
        preferred_l = _coplot_luminance_from_rgba(mcolors.to_rgba(preferred))
        bg_l = _coplot_luminance_from_rgba(rgba)
        lighter = max(preferred_l, bg_l)
        darker = min(preferred_l, bg_l)
        ratio = (lighter + 0.05) / (darker + 0.05)
        if ratio >= 4.5:
            return preferred
    except Exception:
        pass
    bg_l = _coplot_luminance_from_rgba(rgba)
    return '#111111' if bg_l > 0.5 else '#FFFFFF'

def visualize_individual_run(history, run_name):
    epochs = history['epoch']

    # Create figure and axis with publication-quality settings
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=300)
    fig.patch.set_facecolor('#1E293B')

    # Colors
    colors = ["#C8C0F7","#8C7BEF","#2000DF","#1B00BC","#16009C"]

    for ax in axes:
        ax.set_facecolor('#1E293B')
        ax.grid(axis='both', alpha=0.7, color='#5D6674', linestyle='--', linewidth=0.5, zorder=0)
        ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_linewidth(1)
        ax.spines['left'].set_linewidth(1)
        ax.spines['bottom'].set_color('#F1F5F9')
        ax.spines['left'].set_color('#F1F5F9')
        ax.spines['left'].set_position(('outward', 2))
        ax.spines['bottom'].set_position(('outward', 2))
        ax.set_xlabel('Epochs', fontsize=12, labelpad=10, color='#F1F5F9')

    # Accuracy Chart
    axes[0].plot(epochs, history['test_acc'], color=colors[0], linewidth=2, label='Test Acc')
    axes[0].plot(epochs, history['train_acc'], color=colors[1], linewidth=2, linestyle='--', label='Train Acc')
    axes[0].set_ylabel('Accuracy', fontsize=12, color='#F1F5F9', labelpad=10)
    axes[0].set_title(f'Accuracy: {run_name}', color='#F1F5F9')
    axes[0].legend(frameon=False, fontsize=10)

    # Loss Chart
    axes[1].plot(epochs, history['test_loss'], color=colors[0], linewidth=2, label='Test Loss')
    axes[1].plot(epochs, history['train_loss'], color=colors[1], linewidth=2, linestyle='--', label='Train Loss')
    axes[1].set_ylabel('Loss', fontsize=12, color='#F1F5F9', labelpad=10)
    axes[1].set_title(f'Loss: {run_name}', color='#F1F5F9')
    axes[1].legend(frameon=False, fontsize=10)

    plt.tight_layout()
    plt.show()


In [ ]:
#@title (5) model_c utilities
import os

def invoke_model_with_a_b_c(model_c, a, b, c):
    # Ensure the model_c is in evaluation mode
    model_c.eval()

    # Encode input using your one_hot helper function
    # In dataset: c is in range(0, 19) mapping to modulo (c+1), so for modulo C we use index C-1
    xx = np.concatenate([one_hot(a), one_hot(b), one_hot(c-1, 19)])
    xx_tensor = torch.tensor(xx, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    # Perform prediction
    with torch.no_grad():
        output = model_c(xx_tensor)
        prediction = output.argmax(dim=1).item()

    print(f"Prediction for ({a} * {b}) % {c} = {prediction}")
    return prediction

def _clean_state_dict(checkpoint):
    # Clean potential _orig_mod. prefixes caused by torch.compile()
    clean_checkpoint = {}
    for k, v in checkpoint.items():
        if k.startswith("_orig_mod."):
            clean_checkpoint[k.replace("_orig_mod.", "")] = v
        else:
            clean_checkpoint[k] = v
    return clean_checkpoint

def save_model_local(model_c, filename_c='DDN-v2.pth', save_dir_c = './'):
    os.makedirs(save_dir_c, exist_ok=True)
    pathname = f"{save_dir_c}/{filename_c}"

    cleaned_state_dict = _clean_state_dict(model_c.state_dict())
    torch.save(cleaned_state_dict, pathname)
    print(f"model weights saved locally to: {pathname}")
    return pathname

def load_model_local(model_c, pathname, device_c=DEVICE):
    checkpoint = torch.load(pathname, map_location=device_c)
    #clean_checkpoint = _clean_state_dict(checkpoint)
    model_c.load_state_dict(checkpoint)
    print("model weights loaded successfully into a new model instance.")
    return model_c

def save_load_model_local(model_c, new_model_c, filename_c='DDN-v2.pth', save_dir_c = './'):
    # 1. Clean the state dict and save the model_c locally
    pathname = save_model_local(model_c, filename_c, save_dir_c)
    # 2. Instantiate a new model_c and load the state dict
    return load_model_local(new_model_c, pathname)

def save_to_google_drive(model_c, filename_c="DDN-v2.pth", save_dir_c="workshop"):
  try:
    from google.colab import drive

    if not os.path.exists('/content/drive/MyDrive'):
      drive.mount('/content/drive')

    full_save_dir = f"/content/drive/MyDrive/{save_dir_c}"
    os.makedirs(full_save_dir, exist_ok=True)
    pathname = f"{full_save_dir}/{filename_c}"

    cleaned_state_dict = _clean_state_dict(model_c.state_dict())
    torch.save(cleaned_state_dict, pathname)
    print(f"Model saved to Google Drive: {pathname}")

    return pathname
  except Exception as e:
    print(f"Error saving model to Google Drive: {e}")

def load_file_hugging_face(filename_c, repo_id="howardyang2009/llm-workshop-v1"):
    from huggingface_hub import hf_hub_download
    try:
        # Retrieve the HF_TOKEN securely from Colab's secrets manager
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        print("Warning: HF_TOKEN secret not found in Colab secrets. Attempting unauthenticated download.")
        hf_token = None
    pathname = hf_hub_download(repo_id=repo_id, filename=filename_c, token=hf_token)
    print(f"model downloaded from HF to: {pathname}")
    return pathname

def load_model_hugging_face(new_model_c, filename_c="DDN-v2.pth", repo_id="howardyang2009/llm-workshop-v1"):
    # 1. Download the weights from Hugging Face Hub
    pathname = load_file_hugging_face(filename_c, repo_id)
    # 2. Instantiate and load the model_c
    return load_model_local(new_model_c, pathname)